# yolov8n-voc on voc — training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/voc/yolov8n-voc_train.ipynb)

Role: training; tier **N** (`EPOCHS` epochs of `tools/train_v5.py`, days on CPU: [long runs](../../docs/tasks/long-runs.md)); final evaluation on `SUBSET` images. Tasks: [T35.6](../../docs/tasks/M35-yolov8n.md);
notebooks: [M14](../../docs/tasks/M14-notebooks.md).

Prerequisites:
- data: `data/VOCdevkit/VOC2007/ImageSets/Main/test.txt` — tools/get_voc.sh
- weights: `weights/ultralytics/yolov8n.pt` — Ultralytics checkpoint, AGPL-3.0, local only (neither on Drive nor fetched by tools/get_weights.sh): copy it to weights/ultralytics/ (Colab: upload it)

Each step prints the `python tools/…` command it runs: a result can be reproduced
from the command line. Outputs in `build/notebooks/voc/yolov8n-voc/`, never in `results/` (an mAP on `SUBSET`
images is not publishable, M12 rules). Notebook generated by
`python -m tools.notebooks` (`make notebooks`): edit the template
`tools/notebooks/gabarits.py`, not this file.

## Parameters

In [ ]:
NET          = 'yolov8n-voc'  # name in CFG_FILES or cfg (relative to the root)
DATASET      = 'voc'  # key of DATASETS
SPLIT        = None  # None: evaluation split of the dataset
WEIGHTS      = None  # None: EMA weights of OUT (final.npz), evaluated at the end; or a path
DEVICE       = 'gpu'  # cpu | gpu (training only; the rest on CPU)
SUBSET       = 50  # images evaluated; 0: full split (tier N)
SIZE         = None  # input WxH (e.g. '640x192'); None: the one of the cfg
RESIZE       = 'v5'  # v5 (letterbox and thresholds of YOLOv5, T33.4)
PRESET       = 'v8'  # eval_voc preset with RESIZE v5: NMS 0.7 of YOLOv8 (T35.3)
METRIC       = None  # None: the one of the dataset (coco for coco and flir)
DATA_ROOT    = None  # dataset root; None: data/<dataset> (Colab: Drive folder)
INIT         = 'coco'  # coco (heads reinitialized) | he
EPOCHS       = 30  # epochs of the recipe (warmup 3, linear lr)
ITERS        = 0  # stop after n batches (0: all epochs; a short try: 20)
BATCH        = 16  # batch (gradients accumulated to 64)
LR           = 0.01  # lr0 (final lr: 0.01·lr0)
SIZE_TRAIN   = 640  # square training input
TRAIN_SUBSET = 0  # first n training images (0: all)
WORKERS      = 4
DRIVE_DIR    = '/content/drive/MyDrive/EmbeddedCV'  # Colab: archives data/<dataset>.tar, outputs and resume in runs/; None: no Drive
JOBS         = None  # evaluation processes; None: one per CPU core, capped by the memory (Colab TPU runtime: dozens)
SHOW         = 6  # images shown
OUT          = 'build/notebooks/voc/yolov8n-voc'  # notebook outputs
REPO_URL     = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab: cloned repository
REV          = 'main'  # Colab: revision

## Environment

Local: nothing is installed or downloaded, the cell checks the prerequisites and
stops with the command to run. Colab: clone of the repository, installation, CuPy if
`DEVICE = "gpu"`, weights, then data by `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<dataset>.tar` or direct download; ExDark and FLIR: archive only,
made on the PC by `tools/get_datasets.sh pack <dataset>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone of an earlier session: reset to REV (notebook and code of the same commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules imported before the update
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # CPU runtime: no driver, CuPy would say "cudaErrorInsufficientDriver".
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("Colab runtime without a GPU: create a new Colab server "
                               "of GPU type (T4, L4 or A100), or DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # driver version
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("repository root not found: open the notebook from notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
env.prepare(DATASET, 'weights/ultralytics/yolov8n.pt' if INIT == 'coco' else (), None, DATA_ROOT, COLAB, hint='',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Fine-tuning

`tools/train_v5.py` in a subprocess: SGD Nesterov with three parameter groups,
warmup then linear lr, gradients accumulated to a nominal batch of 64, weight EMA,
mosaic / random perspective / HSV / flip, heads reinitialized from the COCO weights
with the bias prior of the reference (YOLOv7-tiny: the OTA loss of
`hyp.scratch.tiny.yaml`). A `checkpoint.npz` is written every 200 batches; running
the cell again resumes (`--resume`).

YOLOv8n (T35.6): v8 loss (task-aligned assigner, BCE + CIoU + DFL), `optimizer=auto`
(AdamW or SGD by the rule of the reference, written to `optimizer.txt`), mosaic off
for the last 10 epochs.

In [ ]:
import csv

from IPython.display import Markdown, display

RUN = Path(OUT)
RUN.mkdir(parents=True, exist_ok=True)
resume = (RUN / "checkpoint.npz").exists()
C.run(C.cmd_train_v5(NET, RUN, DATASET, INIT, resume, EPOCHS, ITERS, BATCH, SIZE_TRAIN,
                     WORKERS, DEVICE, DATA_ROOT, TRAIN_SUBSET, LR), log=RUN / "log.txt")

## Losses

`loss.csv`, per batch and per image: box, objectness and class terms (YOLOv5n,
YOLOv7-tiny), box, class and DFL terms (YOLOv8n). Last rows, then one curve per term.

In [ ]:
rows = list(csv.DictReader((RUN / "loss.csv").open())) if (RUN / "loss.csv").exists() else []
parts = [k for k in ("box", "obj", "cls", "dfl") if rows and k in rows[0]]
for r in rows[-10:]:
    print(", ".join(f"{k} {r[k]}" for k in ("it", "lr", "loss", *parts)))
if rows:
    import matplotlib.pyplot as plt

    it = [int(r["it"]) for r in rows]
    fig, axes = plt.subplots(1, len(parts) + 1, figsize=(4 * (len(parts) + 1), 3))
    for ax, k in zip(axes, ("loss", *parts)):
        ax.plot(it, [float(r[k]) for r in rows], lw=0.8)
        ax.set_title(k)
        ax.set_xlabel("batch")
    fig.tight_layout()
    plt.show()

## Evaluation

The EMA weights by `tools/eval_voc.py --preset v5` (T33.4) on `SUBSET` images.

In [ ]:
result = RUN / "map_float.md"
result.unlink(missing_ok=True)
C.run(C.cmd_eval_voc(NET, WEIGHTS or RUN / "final.npz", DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE,
                     DATA_ROOT, out=RUN / "dets", markdown=result, jobs=JOBS, preset=PRESET))
display(Markdown(result.read_text()))

## Summary

In [ ]:
if (RUN / "optimizer.txt").exists():
    print((RUN / "optimizer.txt").read_text().strip())
print(f"outputs: {RUN}/ (checkpoint.npz, loss.csv, final weights); revision {TRACE['rev']}")
print("equivalent commands:")
for c in C.HISTORY:
    print("  " + c)